# Task 3: Soft MoE Optuna Search

Hyperparameter optimization for soft Mixture-of-Experts (Plan 6 §3.5).
Tunes fine-tuning learning rate, softmax temperature, classification weight $\lambda_3$, and balance weight $\lambda_4$.

In [ ]:
# 1. Config
N_TRIALS = 15
OPTUNA_EPOCHS = 12
STUDY_NAME = "task3_soft_moe"
STORAGE_DB = "sqlite:///task3_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier, SoftMoERestorer
from src.models_vae import ConvVAE
from src.train_loop import train_one_epoch_moe, validate_moe, set_seed

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Objective Function
train_dataset = PetDataset(mode="train", corruption_mode="all", return_label=True, seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", return_label=True, seed=SEED)

def objective(trial: optuna.Trial) -> float:
    finetune_lr = trial.suggest_float("finetune_lr", 1e-5, 1e-4, log=True)
    temp = trial.suggest_float("temperature", 0.5, 2.0)
    lambda3 = trial.suggest_float("lambda3", 0.01, 0.5)
    lambda4 = trial.suggest_float("lambda4", 1e-3, 1e-1, log=True)

    lambdas = {"lambda1": 0.8, "lambda2": 0.2, "lambda3": lambda3, "lambda4": lambda4}

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    moe = SoftMoERestorer(
        gate=CorruptionClassifier(),
        specialist_salt=ConvVAE(),
        specialist_blur=ConvVAE(),
        specialist_occlusion=ConvVAE(),
        temperature=temp
    ).to(device)

    opt = torch.optim.Adam(moe.parameters(), lr=finetune_lr)
    best_loss = float("inf")

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_moe(moe, train_loader, opt, device, lambdas)
        vl = validate_moe(moe, val_loader, device, lambdas)
        if vl["val_loss"] < best_loss:
            best_loss = vl["val_loss"]
        trial.report(vl["val_loss"], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_loss

In [ ]:
# 4. Optimization & Export
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

cfg_path = CONFIGS_ROOT / "task3_best_config.yaml"
with open(cfg_path, "w") as f:
    yaml.dump({
        "task": "task3_soft_moe",
        "best_trial_number": int(study.best_trial.number),
        "best_value": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)

print(f"Best MoE config saved to {cfg_path}")